In [0]:
%run ./common/_utils_config

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

raw_prices = (
    spark.read.option("multiLine", "true").json(f"{raw_path}/prices")
    .select("included", F.col("_metadata.file_modification_time").alias("file_ts"))
)

latest_file = Window.partitionBy("indicator", "price_ts").orderBy(F.col("file_ts").desc())

df_silver_prices = (
    raw_prices
    .select("file_ts", F.explode("included").alias("item"))
    .select("file_ts", F.col("item.attributes.title").alias("indicator"), F.explode("item.attributes.values").alias("v"))
    .select(
        "file_ts",
        "indicator",
        F.to_timestamp(F.substring("v.datetime", 1, 19)).alias("price_ts"),
        F.col("v.value").cast("double").alias("price_eur_mwh"),
    )
    .withColumn("rn", F.row_number().over(latest_file))
    .filter("rn = 1")
    .drop("rn", "file_ts")
)

df_silver_prices.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{schema}.silver_electricity_prices")
display(spark.table(f"{catalog}.{schema}.silver_electricity_prices").groupBy("indicator").count())

In [0]:
df_gold_prices = (
    spark.table(f"{catalog}.{schema}.silver_electricity_prices")
    .groupBy("price_ts")
    .agg(
        F.max(F.when(F.lower("indicator").contains("pvpc"), F.col("price_eur_mwh"))).alias("pvpc_eur_mwh"),
        F.max(F.when(F.lower("indicator").contains("spot"), F.col("price_eur_mwh"))).alias("spot_eur_mwh"),
    )
    .withColumn("price_date", F.to_date("price_ts"))
    .withColumn("hour", F.hour("price_ts"))
    .withColumn("weekday", F.date_format("price_ts", "EEEE"))
    .withColumn("pvpc_eur_kwh", F.round(F.col("pvpc_eur_mwh") / 1000, 4))
    .withColumn("spot_eur_kwh", F.round(F.col("spot_eur_mwh") / 1000, 4))
)

df_gold_prices.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{schema}.gold_hourly_prices")
display(spark.table(f"{catalog}.{schema}.gold_hourly_prices").orderBy(F.col("price_ts").desc()).limit(24))

In [0]:
raw_generation = (
    spark.read.option("multiLine", "true").json(f"{raw_path}/generation")
    .select("included", F.col("_metadata.file_modification_time").alias("file_ts"))
)

latest_gen = Window.partitionBy("technology", "generation_date").orderBy(F.col("file_ts").desc())

df_gold_generation = (
    raw_generation
    .select("file_ts", F.explode("included").alias("item"))
    .select("file_ts", F.col("item.attributes.title").alias("technology"), F.explode("item.attributes.values").alias("v"))
    .select(
        "file_ts",
        "technology",
        F.to_date(F.substring("v.datetime", 1, 10)).alias("generation_date"),
        F.col("v.value").cast("double").alias("generation_mwh"),
    )
    .filter(~F.lower("technology").contains("total"))
    .withColumn("rn", F.row_number().over(latest_gen))
    .filter("rn = 1")
    .drop("rn", "file_ts")
)

df_gold_generation.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{schema}.gold_daily_generation")
display(spark.table(f"{catalog}.{schema}.gold_daily_generation").orderBy(F.col("generation_date").desc()).limit(20))